In [ ]:
%pip install pandas scipy h5py

In [ ]:
import numpy as np
import pandas as pd
import h5py
import time
import os
from scipy.signal import butter, filtfilt

# Parámetros
fs = 1_000_000
n = 1_000_000
t = np.arange(n) / fs

# Filtro Butterworth 4º orden pasabajos fc=10kHz
fc = 10_000
b, a = butter(4, fc / (fs / 2), btype='low')

def filtrar_senal(senal):
    return filtfilt(b, a, senal)

# Señales de ejemplo 
def generar_senales():
    ch1 = np.sin(2 * np.pi * 1000 * t) + 0.05 * np.random.randn(n).astype(np.int16)
    ch2 = np.cos(2 * np.pi * 2000 * t) + 0.05 * np.random.randn(n).astype(np.int16)
    return ch1, ch2

# Generar y guardar todos los formatos
def crear_archivos_base():
    ch1, ch2 = generar_senales()
    interleaved = np.vstack([ch1, ch2]).T  # shape (n,2)

    # CSV
    if not os.path.exists("senales.csv"):
        pd.DataFrame(interleaved, columns=["ch1","ch2"]).to_csv("senales.csv", index=False)

    # NPZ
    if not os.path.exists("senales.npz"):
        np.savez("senales.npz", ch1=ch1, ch2=ch2)

    # HDF5
    if not os.path.exists("senales.h5"):
        with h5py.File("senales.h5", "w") as f:
            f.create_dataset("ch1", data=ch1, compression="gzip")
            f.create_dataset("ch2", data=ch2, compression="gzip")
            f.attrs["fs"] = fs

    # RAW (float32 interleaved)
    if not os.path.exists("senales.raw"):
        interleaved.astype(np.float32).tofile("senales.raw")

    # BIN (float64 interleaved)
    if not os.path.exists("senales.bin"):
        interleaved.astype(np.float64).tofile("senales.bin")

# Lecturas y escrituras por formato

def leer_csv():
    df = pd.read_csv("senales.csv")
    return df.values.T

def escribir_csv(datos):
    pd.DataFrame(datos.T, columns=['ch1','ch2']).to_csv("senales_filtradas.csv", index=False)

def leer_npz():
    with np.load("senales.npz") as f:
        return np.vstack([f['ch1'], f['ch2']])
    
def escribir_npz(datos):
    np.savez("senales_filtradas.npz", ch1=datos[0], ch2=datos[1])

def leer_h5():
    with h5py.File("senales.h5", "r") as f:
        return np.vstack([f["ch1"][:], f["ch2"][:]])
    
def escribir_h5(datos):
    with h5py.File("senales_filtradas.h5", "w") as f:
        f.create_dataset("ch1", data=datos[0], compression="gzip")
        f.create_dataset("ch2", data=datos[1], compression="gzip")

def leer_raw():
    data = np.fromfile("senales.raw", dtype=np.float32).reshape(-1, 2).T
    return data

def escribir_raw(datos):
    datos.astype(np.float32).T.tofile("senales_filtradas.raw")

def leer_bin():
    data = np.fromfile("senales.bin", dtype=np.int16).reshape(-1, 2).T
    return data

def escribir_bin(datos):
    datos.astype(np.float64).T.tofile("senales_filtradas.bin")

# Función auxiliar para medir tiempos de lectura, procesamiento y escritura
def medir_tiempos(formato, leer_func, escribir_func):
    # Lectura
    t0 = time.time()
    datos = leer_func()
    t1 = time.time()
    # Procesamiento
    datos_filtrados = np.vstack([filtrar_senal(ch) for ch in datos])
    t2 = time.time()
    # Escritura
    escribir_func(datos_filtrados)
    t3 = time.time()
    return formato, t1 - t0, t2 - t1, t3 - t2

# --- Flujo principal ---
crear_archivos_base()

for i in range(10):
    pruebas = [
        ("CSV", leer_csv, escribir_csv, "senales_filtradas.csv"),
        ("NPZ", leer_npz, escribir_npz, "senales_filtradas.npz"),
        ("HDF5", leer_h5, escribir_h5, "senales_filtradas.h5"),
        ("RAW", leer_raw, escribir_raw, "senales_filtradas.raw"),
        ("BIN", leer_bin, escribir_bin, "senales_filtradas.bin"),
    ]

    resultados = []
    for formato, leer, escribir, archivo_salida in pruebas:
        print(f"Procesando formato: {formato}")
        r = medir_tiempos(formato, leer, escribir)
        try:
            tam = os.path.getsize(archivo_salida) / (1024**2)
        except OSError:
            tam = float("nan")

        formato_r, t_lect, t_proc, t_escr = r
        rdict = {
            "Formato": formato_r,
            "Tiempo_lectura": t_lect,
            "Tiempo_procesamiento": t_proc,
            "Tiempo_escritura": t_escr,
            "Tamanio_MB_salida": tam
        }
        resultados.append(rdict)

    # --- Mostrar resultados ---
    #import pprint
    #pprint.pprint(resultados)
    # También lo podés volcar a DataFrame si preferís:
    import pandas as pd
    df = pd.DataFrame(resultados)
    print(df.to_string(index=False))


Procesando formato: CSV
Procesando formato: NPZ
Procesando formato: HDF5
Procesando formato: RAW
Procesando formato: BIN
Formato  Tiempo_lectura  Tiempo_procesamiento  Tiempo_escritura  Tamanio_MB_salida
    CSV        0.775790              0.112395          5.372535          38.397155
    NPZ        0.047471              0.084106          0.039206          15.259264
   HDF5        0.162491              0.086557          0.879021          14.543279
    RAW        0.007042              0.104455          0.212831           7.629395
    BIN        0.011355              0.095970          0.263618          15.258789
Procesando formato: CSV
Procesando formato: NPZ
Procesando formato: HDF5
Procesando formato: RAW
Procesando formato: BIN
Formato  Tiempo_lectura  Tiempo_procesamiento  Tiempo_escritura  Tamanio_MB_salida
    CSV        0.767554              0.084461          5.243327          38.397155
    NPZ        0.045320              0.089895          0.041623          15.259264
   HDF5    